# Module B1/B2: Customer Review Sentiment Analysis Setup (TF-IDF & Logistic Regression)

Train a TF-IDF vectorizer and Logistic Regression / Naive Bayes classifier on retail customer feedback reviews (`data/reviews.csv`), preprocess text (lowercasing, punctuation stripping, tokenization, stop-words removal), evaluate classification performance, and export the trained model pipeline to `app/models/sentiment_model.pkl`.

In [1]:
import os
import re
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

MODELS_DIR = "../app/models" if os.path.basename(os.getcwd()) == "notebooks" else "app/models"
DATA_DIR = "../data" if os.path.basename(os.getcwd()) == "notebooks" else "data"

os.makedirs(MODELS_DIR, exist_ok=True)
print(f"[INFO] Target models directory: {os.path.abspath(MODELS_DIR)}")

[INFO] Target models directory: c:\smart-retail-ai\app\models


In [2]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

reviews_csv = os.path.join(DATA_DIR, "reviews.csv")
if os.path.exists(reviews_csv):
    df = pd.read_csv(reviews_csv)
    print(f"[SUCCESS] Loaded {len(df)} customer review entries from {reviews_csv}.")
else:
    # Synthetic benchmark backup data
    data = [
        ("Outstanding quality and super fast delivery! Highly recommended.", "Positive"),
        ("Great store experience, friendly staff and easy checkout.", "Positive"),
        ("The product arrived damaged and customer support was unhelpful.", "Negative"),
        ("Poor quality material. Fits poorly and faded after one wash.", "Negative"),
        ("Decent item for the price. Fits fine.", "Neutral"),
        ("Average customer service. Nothing special.", "Neutral")
    ]
    df = pd.DataFrame(data, columns=["Review Text", "Sentiment"])
    print(f"[INFO] Initialized benchmark review dataset ({len(df)} samples).")

df["cleaned_text"] = df.iloc[:, 0].apply(preprocess_text)
df.head()

[SUCCESS] Loaded 23486 customer review entries from ../data\reviews.csv.


,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name,cleaned_text
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates,
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses,
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses,
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants,
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses,


In [ ]:
print("[INFO] Extracting TF-IDF Features & Fitting Model...")
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df["cleaned_text"])
y = df.iloc[:, 1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(f"Sentiment Classification Accuracy: {accuracy_score(y_test, y_pred):.4f}")

[INFO] Extracting TF-IDF Features & Fitting Model...


ValueError: empty vocabulary; perhaps the documents only contain stop words

: 

In [ ]:
model_path = os.path.join(MODELS_DIR, "sentiment_model.pkl")
joblib.dump({"vectorizer": vectorizer, "model": clf}, model_path)
print(f"[SUCCESS] Exported trained NLP sentiment model pipeline to {model_path}")